# AROON Indicator Strategy Backtesting Framework
## FTMO-Compatible Stock Assets | Production-Ready Research Notebook

---

**Version**: 1.0 | **Last Updated**: 2025-02

### Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | **Theory & Mechanics** | AROON math, interpretation, signal generation |
| 2 | **Data & Setup** | yfinance data pipeline, validation, universe definition |
| 3 | **Indicator Implementation** | From-scratch AROON calculation with visualizations |
| 4 | **Strategy Development** | Crossover, Threshold, and Oscillator strategies |
| 5 | **Backtesting Framework** | Vectorized engine with FTMO constraints |
| 6 | **Performance Analysis** | Equity curves, heatmaps, distributions, metrics |
| 7 | **Parameter Optimization** | Grid search, walk-forward, surface plots |
| 8 | **Robustness Checks** | Multi-asset, regime, Monte Carlo analysis |
| 9 | **Summary & Insights** | Actionable conclusions and next steps |

> **Disclaimer**: This notebook is for **educational and research purposes only**. Past performance does not guarantee future results. Always validate strategies with paper trading before risking capital. This is not financial advice.


## 0. Global Configuration

All tunable parameters are defined here. Change these values to customize the entire analysis without touching downstream code.


In [ ]:
# ==============================================================================
# GLOBAL CONFIGURATION
# ==============================================================================

CONFIG = {
    # -- Universe & Data -------------------------------------------------------
    "tickers": ["SPY", "QQQ", "AAPL", "TSLA", "MSFT", "AMZN", "NVDA", "META"],
    "primary_ticker": "SPY",
    "start_date": "2019-01-01",
    "end_date": "2025-01-31",

    # -- AROON Parameters ------------------------------------------------------
    "aroon_period": 25,
    "aroon_periods_grid": [10, 14, 21, 25, 30, 50],

    # -- Strategy Thresholds ---------------------------------------------------
    "strong_trend_threshold": 70,
    "weak_trend_threshold": 30,

    # -- Backtesting -----------------------------------------------------------
    "initial_capital": 100_000,
    "commission_pct": 0.0,
    "slippage_bps": 5,
    "position_size_pct": 1.0,

    # -- FTMO Constraints ------------------------------------------------------
    "ftmo_max_daily_loss_pct": 0.05,
    "ftmo_max_total_loss_pct": 0.10,
    "ftmo_profit_target_pct": 0.10,

    # -- Walk-Forward & Optimization -------------------------------------------
    "wf_num_folds": 4,

    # -- Monte Carlo -----------------------------------------------------------
    "mc_simulations": 1_000,
    "mc_confidence_level": 0.95,

    # -- Visualization ---------------------------------------------------------
    "fig_width": 14,
    "fig_height": 6,
    "color_palette": {
        "up": "#2ecc71", "down": "#e74c3c", "neutral": "#95a5a6",
        "equity": "#3498db", "benchmark": "#7f8c8d", "drawdown": "#e74c3c",
        "buy": "#27ae60", "sell": "#c0392b",
    }
}

print(f"Configuration loaded | Universe: {CONFIG['tickers']}")
print(f"Period: {CONFIG['start_date']} to {CONFIG['end_date']} | AROON period: {CONFIG['aroon_period']}")


In [ ]:
# ==============================================================================
# IMPORTS
# ==============================================================================
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from typing import Dict, List, Tuple, Optional, Callable
import warnings
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (CONFIG["fig_width"], CONFIG["fig_height"]),
    "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 11,
    "figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
})
np.random.seed(42)
print("All imports loaded")


---
## 1. AROON Indicator: Theory & Mechanics

### 1.1 What Is the AROON Indicator?

The **AROON indicator**, developed by Tushar Chande in 1995, measures the **time elapsed since the most recent high or low** within a lookback window. Unlike momentum oscillators that measure price magnitude, AROON focuses on **when** extremes occurred, making it a *time-based* trend detection tool.

The name "Aroon" comes from Sanskrit, meaning **"dawn's early light"**, reflecting its purpose: to detect trends early.

### 1.2 Mathematical Formulas

Given a lookback period $n$ (default 25 days):

$$\text{AROON-Up}_t = \frac{n - (\text{periods since highest high in last } n \text{ periods})}{n} \times 100$$

$$\text{AROON-Down}_t = \frac{n - (\text{periods since lowest low in last } n \text{ periods})}{n} \times 100$$

$$\text{AROON Oscillator}_t = \text{AROON-Up}_t - \text{AROON-Down}_t$$

**Range**: AROON-Up and AROON-Down oscillate between **0** and **100**. The Oscillator ranges from **-100** to **+100**.

### 1.3 Interpretation Guide

| AROON-Up | AROON-Down | Market Condition |
|----------|-----------|------------------|
| > 70 | < 30 | **Strong uptrend**: recent high is very recent, recent low is distant |
| < 30 | > 70 | **Strong downtrend**: recent low is very recent, recent high is distant |
| Both > 70 | Both > 70 | **Consolidation**: both extremes occurred recently |
| Both < 30 | Both < 30 | **Consolidation**: no recent extremes, range-bound market |

### 1.4 Key Trading Signals

1. **Crossover Signals**: When AROON-Up crosses above AROON-Down = bullish; reverse = bearish
2. **Extreme Readings**: AROON-Up > 70 with AROON-Down < 30 confirms strong uptrend
3. **Oscillator Zero-Cross**: Above zero = bullish momentum; below zero = bearish
4. **Parallel Movement**: Both lines moving together suggests consolidation, avoid trading

### 1.5 Why AROON for FTMO?

FTMO challenges require **consistent profitability with strict drawdown limits**. AROON is well-suited because it identifies trends early, generates clear binary signals, and its time-based measurement is less susceptible to price spikes than momentum oscillators.


---
## 2. Data Acquisition & Validation

We fetch OHLCV data from Yahoo Finance for our FTMO-compatible universe. These are highly liquid US stocks and ETFs.


In [ ]:
# ==============================================================================
# DATA FETCHING
# ==============================================================================

def fetch_universe_data(tickers, start, end):
    """Download OHLCV data for each ticker. Returns dict of ticker -> DataFrame."""
    data = {}
    for ticker in tickers:
        try:
            df = yf.download(ticker, start=start, end=end, progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df.columns = [c.strip().title() for c in df.columns]
            required = ["Open", "High", "Low", "Close", "Volume"]
            df = df[[c for c in required if c in df.columns]].dropna(subset=["Close"])
            if len(df) > 100:
                data[ticker] = df
                print(f"  {ticker}: {len(df)} rows | {df.index[0].date()} to {df.index[-1].date()}")
            else:
                print(f"  {ticker}: Only {len(df)} rows, skipped")
        except Exception as e:
            print(f"  {ticker}: ERROR {e}")
    return data

print("Downloading market data...\n")
universe_data = fetch_universe_data(CONFIG["tickers"], CONFIG["start_date"], CONFIG["end_date"])
print(f"\nLoaded {len(universe_data)} assets successfully")


### 2.1 Data Validation

Before proceeding, we check for gaps, outliers, and data quality issues that could corrupt backtest results.


In [ ]:
# ==============================================================================
# DATA VALIDATION
# ==============================================================================

def validate_ohlcv(df, ticker):
    """Validate and clean OHLCV data: fill gaps, fix OHLC consistency, flag outliers."""
    issues = []
    n_missing = df.isnull().sum().sum()
    if n_missing > 0:
        issues.append(f"{n_missing} missing values forward-filled")
        df = df.ffill(limit=5).dropna()

    bad_hl = (df["High"] < df["Low"]).sum()
    if bad_hl > 0:
        issues.append(f"{bad_hl} rows with High < Low swapped")
        mask = df["High"] < df["Low"]
        df.loc[mask, ["High", "Low"]] = df.loc[mask, ["Low", "High"]].values

    bad_price = (df["Close"] <= 0).sum()
    if bad_price > 0:
        issues.append(f"{bad_price} non-positive prices dropped")
        df = df[df["Close"] > 0]

    extreme = (df["Close"].pct_change().abs() > 0.50).sum()
    if extreme > 0:
        issues.append(f"{extreme} days with |return| > 50% (flagged)")

    status = "; ".join(issues) if issues else "No issues"
    print(f"  {ticker}: {status}")
    return df

print("Validating data quality...\n")
for ticker in list(universe_data.keys()):
    universe_data[ticker] = validate_ohlcv(universe_data[ticker], ticker)

primary = universe_data[CONFIG["primary_ticker"]].copy()
print(f"\nPrimary asset ({CONFIG['primary_ticker']}): {len(primary)} trading days ready")


---
## 3. AROON Indicator: From-Scratch Implementation

We implement AROON entirely from scratch for full transparency. No black-box libraries.


In [ ]:
# ==============================================================================
# AROON INDICATOR -- Transparent implementation from scratch
# ==============================================================================

def compute_aroon_vectorized(high, low, period=25):
    """
    Vectorized AROON calculation using pandas rolling windows.

    Parameters
    ----------
    high, low : pd.Series
        High and Low price series.
    period : int
        Lookback period (default 25).

    Returns
    -------
    aroon_up, aroon_down, aroon_osc : pd.Series
        AROON-Up (0-100), AROON-Down (0-100), Oscillator (-100 to +100).

    Formula
    -------
    AROON-Up   = ((period - days_since_highest_high) / period) * 100
    AROON-Down = ((period - days_since_lowest_low)  / period) * 100
    AROON Osc  = AROON-Up - AROON-Down
    """
    def _days_since_max(x):
        # Returns periods since the most recent max in the window
        return len(x) - 1 - np.argmax(x[::-1])

    def _days_since_min(x):
        # Returns periods since the most recent min in the window
        return len(x) - 1 - np.argmin(x[::-1])

    roll_high = high.rolling(window=period + 1, min_periods=period + 1)
    roll_low  = low.rolling(window=period + 1, min_periods=period + 1)

    days_since_high = roll_high.apply(_days_since_max, raw=True)
    days_since_low  = roll_low.apply(_days_since_min, raw=True)

    aroon_up   = ((period - days_since_high) / period) * 100
    aroon_down = ((period - days_since_low)  / period) * 100
    aroon_osc  = aroon_up - aroon_down

    aroon_up.name   = "AROON_Up"
    aroon_down.name = "AROON_Down"
    aroon_osc.name  = "AROON_Osc"

    return aroon_up, aroon_down, aroon_osc


# Compute AROON for primary asset
aroon_up, aroon_down, aroon_osc = compute_aroon_vectorized(
    primary["High"], primary["Low"], period=CONFIG["aroon_period"]
)
primary["AROON_Up"]   = aroon_up
primary["AROON_Down"] = aroon_down
primary["AROON_Osc"]  = aroon_osc

print(f"AROON({CONFIG['aroon_period']}) computed for {CONFIG['primary_ticker']}")
print(f"Latest: Up={aroon_up.iloc[-1]:.0f}  Down={aroon_down.iloc[-1]:.0f}  Osc={aroon_osc.iloc[-1]:.0f}")


### 3.1 AROON Visualization

Three-panel chart: price with trend shading, AROON-Up/Down with threshold zones, and Oscillator.


In [ ]:
# ==============================================================================
# AROON VISUALIZATION -- 3-panel overview
# ==============================================================================

def plot_aroon_overview(df, ticker, last_n_days=500):
    """3-panel AROON chart: Price + trend shading, AROON lines, Oscillator."""
    d = df.iloc[-last_n_days:].copy()
    c = CONFIG["color_palette"]

    fig, axes = plt.subplots(3, 1, figsize=(CONFIG["fig_width"], 12),
                             gridspec_kw={"height_ratios": [3, 2, 1.5]}, sharex=True)
    fig.suptitle(f"{ticker} -- AROON({CONFIG['aroon_period']}) Analysis",
                 fontsize=15, fontweight="bold", y=0.95)

    # Panel 1: Price with trend shading
    ax1 = axes[0]
    ax1.plot(d.index, d["Close"], color="#2c3e50", linewidth=1.2, label="Close")
    for i in range(1, len(d)):
        if d["AROON_Up"].iloc[i] > 70 and d["AROON_Down"].iloc[i] < 30:
            ax1.axvspan(d.index[i-1], d.index[i], alpha=0.08, color=c["up"])
        elif d["AROON_Down"].iloc[i] > 70 and d["AROON_Up"].iloc[i] < 30:
            ax1.axvspan(d.index[i-1], d.index[i], alpha=0.08, color=c["down"])
    ax1.set_ylabel("Price ($)")
    ax1.legend(loc="upper left")
    ax1.set_title("Price with AROON Trend Shading (Green=Uptrend, Red=Downtrend)")

    # Panel 2: AROON-Up and AROON-Down
    ax2 = axes[1]
    ax2.plot(d.index, d["AROON_Up"], color=c["up"], linewidth=1.3, label="AROON-Up", alpha=0.9)
    ax2.plot(d.index, d["AROON_Down"], color=c["down"], linewidth=1.3, label="AROON-Down", alpha=0.9)
    ax2.axhline(70, color="gray", linewidth=0.7, linestyle="--", alpha=0.5)
    ax2.axhline(30, color="gray", linewidth=0.7, linestyle="--", alpha=0.5)
    ax2.fill_between(d.index, 70, 100, alpha=0.05, color=c["up"], label="Strong (>70)")
    ax2.fill_between(d.index, 0, 30, alpha=0.05, color=c["down"], label="Weak (<30)")
    ax2.set_ylabel("AROON Value")
    ax2.set_ylim(-5, 105)
    ax2.legend(loc="upper right", ncol=2, fontsize=9)
    ax2.set_title("AROON-Up & AROON-Down")

    # Panel 3: Oscillator
    ax3 = axes[2]
    ax3.fill_between(d.index, d["AROON_Osc"], 0, where=d["AROON_Osc"] >= 0,
                     color=c["up"], alpha=0.4, label="Bullish")
    ax3.fill_between(d.index, d["AROON_Osc"], 0, where=d["AROON_Osc"] < 0,
                     color=c["down"], alpha=0.4, label="Bearish")
    ax3.axhline(0, color="black", linewidth=0.8)
    ax3.set_ylabel("Oscillator")
    ax3.set_ylim(-105, 105)
    ax3.legend(loc="upper right", fontsize=9)
    ax3.set_title("AROON Oscillator (Up - Down)")
    ax3.set_xlabel("Date")

    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.show()

plot_aroon_overview(primary, CONFIG["primary_ticker"], last_n_days=500)


---
## 4. Strategy Development

We implement three distinct AROON-based strategies:

| Strategy | Signal Logic | Strengths |
|----------|-------------|-----------|
| **Crossover** | AROON-Up crosses AROON-Down | Early trend detection |
| **Threshold** | Extreme readings (>70 / <30) | High-conviction entries |
| **Oscillator** | Oscillator crosses zero | Simple, single-line signal |

Signals are shifted by 1 bar to prevent look-ahead bias.


In [ ]:
# ==============================================================================
# STRATEGY SIGNAL GENERATORS
# ==============================================================================

def strategy_crossover(df, period=25, **kwargs):
    """
    AROON Crossover Strategy.
    Long when AROON-Up > AROON-Down, Short when AROON-Down > AROON-Up.
    """
    up, down, _ = compute_aroon_vectorized(df["High"], df["Low"], period)
    signal = pd.Series(0.0, index=df.index)
    signal[up > down]  =  1.0
    signal[up < down]  = -1.0
    return signal


def strategy_threshold(df, period=25, strong=70, weak=30, **kwargs):
    """
    AROON Threshold (Extreme Readings) Strategy.
    Long when AROON-Up > strong AND AROON-Down < weak. Short for reverse.
    Flat in ambiguous zones. Holds position until opposite threshold breached.
    """
    up, down, _ = compute_aroon_vectorized(df["High"], df["Low"], period)
    signal = pd.Series(0.0, index=df.index)

    # Enter on strong readings
    signal[(up > strong) & (down < weak)]  =  1.0
    signal[(down > strong) & (up < weak)]  = -1.0

    # Forward-fill to hold positions
    signal = signal.replace(0, np.nan).ffill().fillna(0)

    # Exit when opposite extreme breaches
    exit_long  = (up < weak) & (signal == 1.0)
    exit_short = (down < weak) & (signal == -1.0)
    signal[exit_long | exit_short] = 0.0
    signal = signal.replace(0, np.nan).ffill().fillna(0)
    return signal


def strategy_oscillator(df, period=25, threshold=0, **kwargs):
    """
    AROON Oscillator Zero-Cross Strategy.
    Long when Oscillator > threshold, Short when Oscillator < -threshold.
    """
    _, _, osc = compute_aroon_vectorized(df["High"], df["Low"], period)
    signal = pd.Series(0.0, index=df.index)
    signal[osc > threshold]  =  1.0
    signal[osc < -threshold] = -1.0
    return signal


# Strategy registry
STRATEGIES = {
    "Crossover":  strategy_crossover,
    "Threshold":  strategy_threshold,
    "Oscillator": strategy_oscillator,
}
print("3 strategies defined: Crossover, Threshold, Oscillator")


### 4.1 Signal Visualization

Where each strategy generates buy/sell signals on the price chart.


In [ ]:
# ==============================================================================
# SIGNAL VISUALIZATION
# ==============================================================================

def plot_strategy_signals(df, strategy_func, strategy_name, last_n_days=400, **kwargs):
    """Plot price chart with buy/sell signal markers."""
    signal = strategy_func(df, period=CONFIG["aroon_period"], **kwargs)
    signal_change = signal.diff().fillna(0)
    buy_signals  = signal_change[signal_change > 0].index
    sell_signals = signal_change[signal_change < 0].index

    d = df.iloc[-last_n_days:]
    c = CONFIG["color_palette"]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(CONFIG["fig_width"], 8),
                                    gridspec_kw={"height_ratios": [3, 1]}, sharex=True)

    ax1.plot(d.index, d["Close"], color="#2c3e50", linewidth=1, alpha=0.8)
    buy_vis = buy_signals[buy_signals >= d.index[0]]
    sell_vis = sell_signals[sell_signals >= d.index[0]]
    ax1.scatter(buy_vis, df.loc[buy_vis, "Close"], marker="^", s=80, color=c["buy"],
                zorder=5, label="Buy Signal")
    ax1.scatter(sell_vis, df.loc[sell_vis, "Close"], marker="v", s=80, color=c["sell"],
                zorder=5, label="Sell Signal")
    ax1.set_ylabel("Price ($)")
    ax1.set_title(f"{CONFIG['primary_ticker']} -- {strategy_name} Strategy Signals")
    ax1.legend(loc="upper left")

    pos = signal.loc[d.index]
    ax2.fill_between(d.index, pos, 0, where=pos > 0, color=c["up"], alpha=0.4, label="Long")
    ax2.fill_between(d.index, pos, 0, where=pos < 0, color=c["down"], alpha=0.4, label="Short")
    ax2.axhline(0, color="black", linewidth=0.5)
    ax2.set_ylabel("Position")
    ax2.set_ylim(-1.3, 1.3)
    ax2.legend(loc="upper right", fontsize=9)
    ax2.set_xlabel(f"Date | Buy: {len(buy_vis)}, Sell: {len(sell_vis)}")
    plt.tight_layout()
    plt.show()

for name, func in STRATEGIES.items():
    plot_strategy_signals(primary, func, name)


---
## 5. Backtesting Framework

Our vectorized backtesting engine incorporates:
- **Look-ahead bias prevention**: Signals shifted by 1 bar
- **Transaction costs**: Slippage modeled in basis points per trade
- **FTMO constraints**: 5% max daily loss, 10% max total drawdown, 10% profit target
- **Trade-by-trade logging**: Full record of every entry and exit


In [ ]:
# ==============================================================================
# BACKTESTING ENGINE -- Vectorized with FTMO constraints
# ==============================================================================

class AroonBacktester:
    """
    Vectorized backtesting engine for AROON strategies.
    Includes look-ahead bias prevention, transaction costs, FTMO limits,
    and trade-by-trade logging.
    """

    def __init__(self, df, initial_capital=100_000, slippage_bps=5,
                 commission_pct=0.0, position_size_pct=1.0):
        self.df = df.copy()
        self.initial_capital = initial_capital
        self.slippage_cost = slippage_bps / 10_000
        self.commission_pct = commission_pct
        self.position_size_pct = position_size_pct
        self.results = None

    def run(self, signal_func, strategy_name="Strategy",
            apply_ftmo_constraints=True, **signal_kwargs):
        """Execute a full backtest. Returns dict with metrics, equity, trades."""
        df = self.df.copy()

        # Generate and shift signals (prevent look-ahead bias)
        raw_signal = signal_func(df, **signal_kwargs)
        signal = raw_signal.shift(1).fillna(0)

        # Calculate returns
        asset_returns = df["Close"].pct_change().fillna(0)

        # Transaction costs on position changes
        position_changes = signal.diff().fillna(0).abs()
        total_cost = position_changes * (self.slippage_cost + self.commission_pct)

        # Net strategy returns
        strat_returns = (signal * asset_returns - total_cost) * self.position_size_pct

        # Apply FTMO constraints if requested
        if apply_ftmo_constraints:
            strat_returns = self._apply_ftmo_limits(strat_returns)

        # Equity curve
        equity = self.initial_capital * (1 + strat_returns).cumprod()

        # Extract trades
        trades = self._extract_trades(signal, df["Close"])

        # Performance metrics
        metrics = self._compute_metrics(strat_returns, equity, trades, strategy_name)

        self.results = {
            "metrics": metrics, "equity_curve": equity, "returns": strat_returns,
            "trades": trades, "signals": signal, "asset_returns": asset_returns,
        }
        return self.results

    def _apply_ftmo_limits(self, returns):
        """Enforce FTMO risk constraints: 5% daily loss, 10% max DD, 10% target."""
        adjusted = returns.copy()
        cum_return = 0.0
        peak = 0.0
        stopped = False

        for i in range(len(adjusted)):
            if stopped:
                adjusted.iloc[i] = 0.0
                continue
            cum_return = (1 + cum_return) * (1 + adjusted.iloc[i]) - 1
            peak = max(peak, cum_return)
            dd = (1 + cum_return) / (1 + peak) - 1 if peak > 0 else cum_return

            if adjusted.iloc[i] < -CONFIG["ftmo_max_daily_loss_pct"]:
                stopped = True
                adjusted.iloc[i] = -CONFIG["ftmo_max_daily_loss_pct"]
            if dd < -CONFIG["ftmo_max_total_loss_pct"]:
                stopped = True
                adjusted.iloc[i] = 0.0
            if cum_return >= CONFIG["ftmo_profit_target_pct"]:
                stopped = True
        return adjusted

    def _extract_trades(self, signal, prices):
        """Build trade-by-trade log from signal series."""
        trades = []
        in_trade = False
        entry_date = entry_price = direction = None

        for i in range(1, len(signal)):
            curr, prev = signal.iloc[i], signal.iloc[i - 1]
            if curr != prev:
                if in_trade:
                    exit_date = signal.index[i]
                    exit_price = prices.iloc[i]
                    ret = (exit_price / entry_price - 1) if direction == 1 else (entry_price / exit_price - 1)
                    trades.append({
                        "entry_date": entry_date, "exit_date": exit_date,
                        "direction": "Long" if direction == 1 else "Short",
                        "entry_price": round(entry_price, 2),
                        "exit_price": round(exit_price, 2),
                        "return_pct": round(ret * 100, 3),
                        "pnl_dollar": round(ret * self.initial_capital * self.position_size_pct, 2),
                        "holding_days": (exit_date - entry_date).days,
                    })
                    in_trade = False
                if curr != 0:
                    entry_date = signal.index[i]
                    entry_price = prices.iloc[i]
                    direction = int(curr)
                    in_trade = True
        return pd.DataFrame(trades) if trades else pd.DataFrame()

    def _compute_metrics(self, returns, equity, trades, name):
        """Compute comprehensive performance statistics."""
        rets = returns.dropna()
        if len(rets) == 0:
            return {"strategy": name, "error": "No returns"}

        cum = (1 + rets).cumprod()
        dd = (cum / cum.expanding().max()) - 1
        ann = 252

        total_return = (1 + rets).prod() - 1
        annual_return = rets.mean() * ann
        annual_vol = rets.std() * np.sqrt(ann)
        sharpe = annual_return / annual_vol if annual_vol > 0 else 0

        downside = rets[rets < 0]
        downside_std = downside.std() * np.sqrt(ann) if len(downside) > 0 else 1e-10
        sortino = annual_return / downside_std

        max_dd = dd.min()
        calmar = annual_return / abs(max_dd) if max_dd != 0 else 0

        if len(trades) > 0:
            wins = trades[trades["return_pct"] > 0]
            losses = trades[trades["return_pct"] <= 0]
            win_rate = len(wins) / len(trades)
            avg_win = wins["return_pct"].mean() if len(wins) > 0 else 0
            avg_loss = losses["return_pct"].mean() if len(losses) > 0 else 0
            gross_profit = wins["pnl_dollar"].sum() if len(wins) > 0 else 0
            gross_loss = abs(losses["pnl_dollar"].sum()) if len(losses) > 0 else 1e-10
            profit_factor = gross_profit / gross_loss
            avg_hold = trades["holding_days"].mean()
        else:
            win_rate = avg_win = avg_loss = profit_factor = avg_hold = 0

        return {
            "strategy": name,
            "total_return_pct": round(total_return * 100, 2),
            "annual_return_pct": round(annual_return * 100, 2),
            "annual_volatility_pct": round(annual_vol * 100, 2),
            "sharpe_ratio": round(sharpe, 3),
            "sortino_ratio": round(sortino, 3),
            "max_drawdown_pct": round(max_dd * 100, 2),
            "calmar_ratio": round(calmar, 3),
            "num_trades": len(trades),
            "win_rate_pct": round(win_rate * 100, 1),
            "avg_win_pct": round(avg_win, 2),
            "avg_loss_pct": round(avg_loss, 2),
            "profit_factor": round(profit_factor, 3),
            "avg_holding_days": round(avg_hold, 1),
            "total_days": len(rets),
        }

print("AroonBacktester engine ready")


### 5.1 Running All Three Strategies

Backtest each strategy on the primary asset and compare results side by side.


In [ ]:
# ==============================================================================
# RUN ALL STRATEGIES
# ==============================================================================

all_results = {}
for name, func in STRATEGIES.items():
    bt = AroonBacktester(
        df=primary, initial_capital=CONFIG["initial_capital"],
        slippage_bps=CONFIG["slippage_bps"], commission_pct=CONFIG["commission_pct"],
        position_size_pct=CONFIG["position_size_pct"],
    )
    result = bt.run(signal_func=func, strategy_name=name,
                    apply_ftmo_constraints=False, period=CONFIG["aroon_period"])
    all_results[name] = result
    m = result["metrics"]
    print(f"  {name}: Return={m['total_return_pct']:+.1f}% | "
          f"Sharpe={m['sharpe_ratio']:.3f} | MaxDD={m['max_drawdown_pct']:.1f}% | "
          f"Trades={m['num_trades']}")

print("\nAll backtests completed")


---
## 6. Performance Analysis & Visualization


In [ ]:
# ==============================================================================
# 6.1 EQUITY CURVES WITH DRAWDOWN OVERLAY
# ==============================================================================

def plot_equity_and_drawdown(all_results, benchmark_df):
    """Equity curves for all strategies + benchmark with drawdown overlay."""
    c = CONFIG["color_palette"]
    colors = ["#3498db", "#e74c3c", "#f39c12", "#9b59b6"]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(CONFIG["fig_width"], 9),
                                    gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
    fig.suptitle(f"{CONFIG['primary_ticker']} -- Strategy Equity Curves & Drawdown",
                 fontsize=14, fontweight="bold")

    # Benchmark
    bm_ret = benchmark_df["Close"].pct_change().fillna(0)
    bm_equity = CONFIG["initial_capital"] * (1 + bm_ret).cumprod()
    ax1.plot(bm_equity.index, bm_equity, color=c["benchmark"],
             linewidth=1.5, linestyle="--", label="Buy & Hold", alpha=0.7)

    for i, (name, res) in enumerate(all_results.items()):
        eq = res["equity_curve"]
        ax1.plot(eq.index, eq, color=colors[i % len(colors)], linewidth=1.3, label=name)
        cum = (1 + res["returns"]).cumprod()
        dd = (cum / cum.expanding().max()) - 1
        ax2.fill_between(dd.index, dd * 100, 0, color=colors[i % len(colors)],
                         alpha=0.25, label=name)

    ax1.set_ylabel("Equity ($)")
    ax1.legend(loc="upper left")
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))
    ax2.set_ylabel("Drawdown (%)")
    ax2.set_xlabel("Date")
    ax2.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.show()

plot_equity_and_drawdown(all_results, primary)


In [ ]:
# ==============================================================================
# 6.2 METRICS COMPARISON TABLE
# ==============================================================================

def display_metrics_table(all_results):
    """Formatted comparison table of all strategy metrics."""
    rows = [res["metrics"] for res in all_results.values()]
    df = pd.DataFrame(rows).set_index("strategy")
    col_map = {
        "total_return_pct": "Total Ret %", "annual_return_pct": "Ann Ret %",
        "annual_volatility_pct": "Ann Vol %", "sharpe_ratio": "Sharpe",
        "sortino_ratio": "Sortino", "max_drawdown_pct": "Max DD %",
        "calmar_ratio": "Calmar", "num_trades": "# Trades",
        "win_rate_pct": "Win %", "avg_win_pct": "Avg Win %",
        "avg_loss_pct": "Avg Loss %", "profit_factor": "PF",
        "avg_holding_days": "Avg Hold", "total_days": "Days",
    }
    df = df.rename(columns=col_map)
    print("=" * 80)
    print("STRATEGY PERFORMANCE COMPARISON")
    print("=" * 80)
    display(df.T)
    return df

metrics_table = display_metrics_table(all_results)


In [ ]:
# ==============================================================================
# 6.3 MONTHLY RETURNS HEATMAP
# ==============================================================================

best_strategy = max(all_results.keys(),
                    key=lambda k: all_results[k]["metrics"]["sharpe_ratio"])

def plot_monthly_heatmap(returns, strategy_name):
    """Monthly returns heatmap (months x years)."""
    monthly = returns.resample("ME").sum() * 100
    mdf = pd.DataFrame({"Year": monthly.index.year, "Month": monthly.index.month,
                         "Return": monthly.values})
    pivot = mdf.pivot_table(index="Year", columns="Month", values="Return", aggfunc="sum")
    pivot.columns = ["Jan","Feb","Mar","Apr","May","Jun",
                     "Jul","Aug","Sep","Oct","Nov","Dec"][:len(pivot.columns)]
    pivot["Annual"] = pivot.sum(axis=1)

    fig, ax = plt.subplots(figsize=(CONFIG["fig_width"], max(3, len(pivot) * 0.6 + 1)))
    cmap = LinearSegmentedColormap.from_list("rg", ["#e74c3c", "#ffffff", "#2ecc71"])
    sns.heatmap(pivot, annot=True, fmt=".1f", center=0, cmap=cmap,
                linewidths=0.5, linecolor="white", ax=ax, annot_kws={"size": 9},
                cbar_kws={"label": "Monthly Return (%)", "shrink": 0.8})
    ax.set_title(f"{strategy_name} -- Monthly Returns Heatmap (%)",
                 fontsize=13, fontweight="bold")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

plot_monthly_heatmap(all_results[best_strategy]["returns"], best_strategy)


In [ ]:
# ==============================================================================
# 6.4 RETURN DISTRIBUTION & TRADE ANALYSIS
# ==============================================================================

def plot_return_distribution(all_results, best_strategy):
    """Daily return histograms and trade P&L distribution."""
    colors = ["#3498db", "#e74c3c", "#f39c12"]
    fig, axes = plt.subplots(1, 2, figsize=(CONFIG["fig_width"], 5))

    ax1 = axes[0]
    for i, (name, res) in enumerate(all_results.items()):
        rets = res["returns"].dropna() * 100
        ax1.hist(rets, bins=80, alpha=0.45, color=colors[i % 3],
                 label=f"{name} (avg={rets.mean():.2f}%)", density=True)
    ax1.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax1.set_xlabel("Daily Return (%)")
    ax1.set_ylabel("Density")
    ax1.set_title("Daily Return Distributions")
    ax1.legend(fontsize=9)

    ax2 = axes[1]
    best = all_results[best_strategy]
    if len(best["trades"]) > 0:
        trade_rets = best["trades"]["return_pct"]
        ax2.hist(trade_rets, bins=40, color="#3498db", alpha=0.7, edgecolor="white")
        ax2.axvline(0, color="black", linewidth=0.8, linestyle="--")
        ax2.axvline(trade_rets.mean(), color="orange", linewidth=1.5, linestyle="--",
                    label=f"Mean: {trade_rets.mean():.2f}%")
        ax2.set_xlabel("Trade Return (%)")
        ax2.set_ylabel("Frequency")
        ax2.set_title(f"{best_strategy} -- Trade Return Distribution")
        ax2.legend()
    plt.tight_layout()
    plt.show()

plot_return_distribution(all_results, best_strategy)


In [ ]:
# ==============================================================================
# 6.5 TRADE LOG
# ==============================================================================

best_trades = all_results[best_strategy]["trades"]
if len(best_trades) > 0:
    print(f"{'='*80}")
    print(f"{best_strategy} Strategy -- Trade Log (first 20 of {len(best_trades)} trades)")
    print(f"{'='*80}")
    display(best_trades.head(20))

    wins = (best_trades["return_pct"] > 0).sum()
    losses = (best_trades["return_pct"] <= 0).sum()
    print(f"\nTrade Summary:")
    print(f"  Total: {len(best_trades)} | Wins: {wins} | Losses: {losses}")
    print(f"  Best: {best_trades['return_pct'].max():+.2f}% | "
          f"Worst: {best_trades['return_pct'].min():+.2f}%")
    print(f"  Avg hold: {best_trades['holding_days'].mean():.1f} days")
else:
    print("No trades generated.")


---
## 7. Parameter Optimization

We perform systematic grid search over AROON periods, then validate with walk-forward analysis to prevent overfitting. Key principle: prefer parameters that produce a *flat* optimization surface (robust) over a *sharp* peak (likely overfit).


In [ ]:
# ==============================================================================
# 7.1 GRID SEARCH
# ==============================================================================

def grid_search_aroon_period(df, strategy_func, periods, strategy_name):
    """Grid search over AROON periods. Returns DataFrame with results per period."""
    results = []
    for period in periods:
        bt = AroonBacktester(df=df, initial_capital=CONFIG["initial_capital"],
                             slippage_bps=CONFIG["slippage_bps"])
        res = bt.run(signal_func=strategy_func, strategy_name=f"{strategy_name}_P{period}",
                     apply_ftmo_constraints=False, period=period)
        m = res["metrics"]
        m["period"] = period
        m["strategy_type"] = strategy_name
        results.append(m)
    return pd.DataFrame(results)

print("Running grid search optimization...\n")
all_opt_results = []
for name, func in STRATEGIES.items():
    opt_df = grid_search_aroon_period(primary, func, CONFIG["aroon_periods_grid"], name)
    all_opt_results.append(opt_df)
    best_idx = opt_df["sharpe_ratio"].idxmax()
    best_row = opt_df.loc[best_idx]
    print(f"  {name:12s} -> Best period: {int(best_row['period'])} "
          f"(Sharpe={best_row['sharpe_ratio']:.3f}, Return={best_row['total_return_pct']:+.1f}%)")

optimization_df = pd.concat(all_opt_results, ignore_index=True)
print("\nGrid search complete")


In [ ]:
# ==============================================================================
# 7.2 OPTIMIZATION SURFACE
# ==============================================================================

def plot_optimization_surface(opt_df):
    """Sharpe, Return, and MaxDD vs AROON period for each strategy."""
    fig, axes = plt.subplots(1, 3, figsize=(CONFIG["fig_width"], 5))
    metrics_plot = [
        ("sharpe_ratio", "Sharpe Ratio", "Higher is better"),
        ("total_return_pct", "Total Return (%)", "Higher is better"),
        ("max_drawdown_pct", "Max Drawdown (%)", "Less negative is better"),
    ]
    colors = {"Crossover": "#3498db", "Threshold": "#e74c3c", "Oscillator": "#f39c12"}

    for ax, (metric, title, note) in zip(axes, metrics_plot):
        for sn in opt_df["strategy_type"].unique():
            sub = opt_df[opt_df["strategy_type"] == sn]
            ax.plot(sub["period"], sub[metric], marker="o", linewidth=2, markersize=6,
                    color=colors.get(sn, "gray"), label=sn)
        ax.set_xlabel("AROON Period")
        ax.set_ylabel(title)
        ax.set_title(f"{title}\n({note})", fontsize=11)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    fig.suptitle("Parameter Optimization Surface", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

plot_optimization_surface(optimization_df)


In [ ]:
# ==============================================================================
# 7.3 WALK-FORWARD ANALYSIS
# ==============================================================================

def walk_forward_analysis(df, strategy_func, periods_to_test, strategy_name, n_folds=4):
    """
    Walk-forward with anchored expanding windows.
    Optimizes on in-sample, tests on out-of-sample for each fold.
    """
    total_len = len(df)
    fold_size = total_len // n_folds
    wf_results = []

    for fold in range(1, n_folds):
        is_end = fold * fold_size
        oos_end = min((fold + 1) * fold_size, total_len)
        is_data = df.iloc[:is_end]
        oos_data = df.iloc[is_end:oos_end]
        if len(oos_data) < 50:
            continue

        # Optimize on in-sample
        best_sharpe, best_period = -np.inf, periods_to_test[0]
        for period in periods_to_test:
            bt = AroonBacktester(df=is_data, initial_capital=CONFIG["initial_capital"],
                                 slippage_bps=CONFIG["slippage_bps"])
            res = bt.run(signal_func=strategy_func, apply_ftmo_constraints=False, period=period)
            if res["metrics"]["sharpe_ratio"] > best_sharpe:
                best_sharpe = res["metrics"]["sharpe_ratio"]
                best_period = period

        # Test on out-of-sample
        bt_oos = AroonBacktester(df=oos_data, initial_capital=CONFIG["initial_capital"],
                                 slippage_bps=CONFIG["slippage_bps"])
        oos_res = bt_oos.run(signal_func=strategy_func, apply_ftmo_constraints=False,
                             period=best_period)
        wf_results.append({
            "fold": fold,
            "is_period": f"{is_data.index[0].date()} to {is_data.index[-1].date()}",
            "oos_period": f"{oos_data.index[0].date()} to {oos_data.index[-1].date()}",
            "best_is_period": best_period,
            "is_sharpe": round(best_sharpe, 3),
            "oos_sharpe": oos_res["metrics"]["sharpe_ratio"],
            "oos_return_pct": oos_res["metrics"]["total_return_pct"],
            "oos_max_dd_pct": oos_res["metrics"]["max_drawdown_pct"],
        })
    return pd.DataFrame(wf_results)

print("Running walk-forward analysis...\n")
wf_all = {}
for name, func in STRATEGIES.items():
    wf_df = walk_forward_analysis(primary, func, CONFIG["aroon_periods_grid"],
                                  name, n_folds=CONFIG["wf_num_folds"])
    wf_all[name] = wf_df
    if len(wf_df) > 0:
        avg_oos = wf_df["oos_sharpe"].mean()
        print(f"  {name:12s} -> Avg OOS Sharpe: {avg_oos:.3f} | "
              f"Periods: {wf_df['best_is_period'].tolist()}")

print("\nWalk-forward analysis complete")


In [ ]:
# Display walk-forward results with overfitting check
for name, wf_df in wf_all.items():
    print(f"\n{'='*70}")
    print(f"Walk-Forward: {name}")
    print(f"{'='*70}")
    display(wf_df)
    if len(wf_df) > 0:
        avg_is = wf_df["is_sharpe"].mean()
        avg_oos = wf_df["oos_sharpe"].mean()
        deg = ((avg_oos - avg_is) / abs(avg_is) * 100) if avg_is != 0 else 0
        status = "Possible overfitting" if deg < -50 else "Reasonable stability"
        print(f"  IS->OOS Sharpe degradation: {deg:.1f}% | {status}")


---
## 8. Robustness Checks

A strategy is only useful if it works beyond a single asset and time period. We test:
1. **Multi-asset performance** across the universe
2. **Market regime analysis** in bull/bear/sideways conditions
3. **Monte Carlo simulation** for confidence intervals


In [ ]:
# ==============================================================================
# 8.1 MULTI-ASSET TESTING
# ==============================================================================

def test_across_universe(universe_data, strategy_func, strategy_name, period):
    """Run same strategy across all assets. Returns DataFrame with metrics per asset."""
    results = []
    for ticker, df in universe_data.items():
        try:
            bt = AroonBacktester(df=df, initial_capital=CONFIG["initial_capital"],
                                 slippage_bps=CONFIG["slippage_bps"])
            res = bt.run(signal_func=strategy_func,
                         strategy_name=f"{strategy_name}_{ticker}",
                         apply_ftmo_constraints=False, period=period)
            m = res["metrics"]
            m["ticker"] = ticker
            results.append(m)
        except Exception as e:
            print(f"  {ticker}: {e}")
    return pd.DataFrame(results)

best_func = STRATEGIES[best_strategy]
print(f"Testing {best_strategy} across {len(universe_data)} assets...\n")
multi_asset_df = test_across_universe(universe_data, best_func, best_strategy,
                                       CONFIG["aroon_period"])

cols = ["ticker", "total_return_pct", "sharpe_ratio", "max_drawdown_pct",
        "win_rate_pct", "profit_factor", "num_trades"]
print(f"{'='*80}")
print(f"Multi-Asset Results: {best_strategy} (AROON={CONFIG['aroon_period']})")
print(f"{'='*80}")
display(multi_asset_df[cols].sort_values("sharpe_ratio", ascending=False))
print(f"\nMedian Sharpe: {multi_asset_df['sharpe_ratio'].median():.3f}")
print(f"Profitable: {(multi_asset_df['sharpe_ratio'] > 0).sum()} / {len(multi_asset_df)}")


In [ ]:
# ==============================================================================
# 8.2 MARKET REGIME ANALYSIS
# ==============================================================================

def regime_analysis(df, strategy_func, period):
    """Classify market into regimes and measure strategy performance in each."""
    d = df.copy()
    sma200 = d["Close"].rolling(200).mean()
    vol60 = d["Close"].pct_change().rolling(60).std() * np.sqrt(252)
    median_vol = vol60.median()

    d["regime"] = "Undefined"
    d.loc[(d["Close"] > sma200) & (vol60 <= median_vol), "regime"] = "Bull (Low Vol)"
    d.loc[(d["Close"] > sma200) & (vol60 > median_vol),  "regime"] = "Bull (High Vol)"
    d.loc[(d["Close"] <= sma200) & (vol60 <= median_vol), "regime"] = "Bear (Low Vol)"
    d.loc[(d["Close"] <= sma200) & (vol60 > median_vol),  "regime"] = "Bear (High Vol)"

    bt = AroonBacktester(df=d, initial_capital=CONFIG["initial_capital"],
                         slippage_bps=CONFIG["slippage_bps"])
    res = bt.run(signal_func=strategy_func, apply_ftmo_constraints=False, period=period)
    strat_rets = res["returns"]

    stats = []
    for regime in d["regime"].unique():
        if regime == "Undefined":
            continue
        mask = d["regime"] == regime
        r = strat_rets[mask].dropna()
        if len(r) < 20:
            continue
        ann_ret = r.mean() * 252
        ann_vol = r.std() * np.sqrt(252)
        sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
        stats.append({"Regime": regime, "Days": len(r),
                       "Ann Return %": round(ann_ret * 100, 2),
                       "Ann Vol %": round(ann_vol * 100, 2),
                       "Sharpe": round(sharpe, 3),
                       "Win Rate %": round((r > 0).mean() * 100, 1)})
    return pd.DataFrame(stats)

print(f"Market Regime Analysis: {best_strategy}\n")
regime_df = regime_analysis(primary, best_func, CONFIG["aroon_period"])
display(regime_df)

# Visualization
if len(regime_df) > 0:
    colors_r = ["#2ecc71", "#27ae60", "#e74c3c", "#c0392b"]
    fig, axes = plt.subplots(1, 2, figsize=(CONFIG["fig_width"], 5))
    axes[0].bar(regime_df["Regime"], regime_df["Sharpe"],
                color=colors_r[:len(regime_df)], edgecolor="white")
    axes[0].axhline(0, color="black", linewidth=0.8)
    axes[0].set_ylabel("Sharpe Ratio")
    axes[0].set_title("Sharpe by Regime")
    axes[0].tick_params(axis="x", rotation=25)

    axes[1].bar(regime_df["Regime"], regime_df["Ann Return %"],
                color=colors_r[:len(regime_df)], edgecolor="white")
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].set_ylabel("Ann. Return (%)")
    axes[1].set_title("Return by Regime")
    axes[1].tick_params(axis="x", rotation=25)
    fig.suptitle(f"{best_strategy} -- Performance by Market Regime",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


In [ ]:
# ==============================================================================
# 8.3 MONTE CARLO SIMULATION
# ==============================================================================

def monte_carlo_simulation(returns, n_simulations=1000, n_days=252,
                           initial_capital=100_000, confidence=0.95):
    """
    Monte Carlo by bootstrap-resampling daily returns.
    Generates n_simulations equity paths over n_days trading days.
    """
    clean = returns.dropna().values
    paths = np.zeros((n_simulations, n_days))
    for i in range(n_simulations):
        paths[i] = np.random.choice(clean, size=n_days, replace=True)

    equity_paths = initial_capital * np.cumprod(1 + paths, axis=1)
    terminal = equity_paths[:, -1]

    max_dds = []
    for path in equity_paths:
        peak = np.maximum.accumulate(path)
        dd = (path - peak) / peak
        max_dds.append(dd.min())
    max_dds = np.array(max_dds)
    alpha = (1 - confidence) / 2

    return {
        "equity_paths": equity_paths, "terminal_values": terminal,
        "max_drawdowns": max_dds,
        "mean_terminal": terminal.mean(), "median_terminal": np.median(terminal),
        "ci_lower": np.percentile(terminal, alpha * 100),
        "ci_upper": np.percentile(terminal, (1 - alpha) * 100),
        "prob_profit": (terminal > initial_capital).mean(),
        "mean_max_dd": max_dds.mean(),
        "worst_dd_95": np.percentile(max_dds, 5),
    }

best_returns = all_results[best_strategy]["returns"]
print(f"Running Monte Carlo ({CONFIG['mc_simulations']} paths)...\n")
mc = monte_carlo_simulation(best_returns, n_simulations=CONFIG["mc_simulations"],
                             n_days=252, initial_capital=CONFIG["initial_capital"],
                             confidence=CONFIG["mc_confidence_level"])

print(f"Monte Carlo Results ({best_strategy}, 1-year horizon):")
print(f"  Prob of profit:    {mc['prob_profit']*100:.1f}%")
print(f"  Median terminal:   ${mc['median_terminal']:,.0f}")
ci_pct = int(CONFIG["mc_confidence_level"] * 100)
print(f"  {ci_pct}% CI:           ${mc['ci_lower']:,.0f} -- ${mc['ci_upper']:,.0f}")
print(f"  Mean max drawdown: {mc['mean_max_dd']*100:.1f}%")
print(f"  Worst-case DD:     {mc['worst_dd_95']*100:.1f}%")


In [ ]:
# ==============================================================================
# 8.3b MONTE CARLO VISUALIZATION
# ==============================================================================

fig, axes = plt.subplots(1, 2, figsize=(CONFIG["fig_width"], 5.5))

# Equity paths fan chart
ax1 = axes[0]
paths = mc["equity_paths"]
x = np.arange(paths.shape[1])
for pct, alpha in [(5, 0.15), (25, 0.2), (50, 0.0)]:
    lower = np.percentile(paths, pct, axis=0)
    upper = np.percentile(paths, 100 - pct, axis=0)
    if pct == 50:
        ax1.plot(x, lower, color="#3498db", linewidth=2, label="Median")
    else:
        ax1.fill_between(x, lower, upper, alpha=alpha, color="#3498db",
                         label=f"{100-2*pct}% CI")
for i in range(min(50, len(paths))):
    ax1.plot(x, paths[i], color="gray", alpha=0.03, linewidth=0.5)
ax1.axhline(CONFIG["initial_capital"], color="black", linewidth=0.8, linestyle="--",
            label="Start")
ax1.set_xlabel("Trading Days")
ax1.set_ylabel("Equity ($)")
ax1.set_title("Monte Carlo Equity Paths (1 Year)")
ax1.legend(fontsize=8, loc="upper left")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))

# Terminal value distribution
ax2 = axes[1]
ax2.hist(mc["terminal_values"], bins=60, color="#3498db", alpha=0.7, edgecolor="white",
         density=True)
ax2.axvline(CONFIG["initial_capital"], color="black", linewidth=1.5, linestyle="--",
            label="Break-even")
ax2.axvline(mc["median_terminal"], color="orange", linewidth=1.5,
            label=f"Median: ${mc['median_terminal']:,.0f}")
ax2.axvline(mc["ci_lower"], color="red", linewidth=1, linestyle=":",
            label=f"5th pctl: ${mc['ci_lower']:,.0f}")
ax2.axvline(mc["ci_upper"], color="green", linewidth=1, linestyle=":",
            label=f"95th pctl: ${mc['ci_upper']:,.0f}")
ax2.set_xlabel("Terminal Value ($)")
ax2.set_ylabel("Density")
ax2.set_title("Distribution of 1-Year Outcomes")
ax2.legend(fontsize=8)
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))

fig.suptitle(f"Monte Carlo -- {best_strategy} ({CONFIG['mc_simulations']:,} sims)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 8.4 FTMO-Constrained Backtest

Re-run the best strategy with FTMO actual constraints enforced: 5% max daily loss, 10% max total drawdown, 10% profit target.


In [ ]:
# ==============================================================================
# FTMO-CONSTRAINED BACKTEST
# ==============================================================================

bt_ftmo = AroonBacktester(df=primary, initial_capital=CONFIG["initial_capital"],
                           slippage_bps=CONFIG["slippage_bps"])
ftmo_result = bt_ftmo.run(signal_func=best_func, strategy_name=f"{best_strategy} (FTMO)",
                           apply_ftmo_constraints=True, period=CONFIG["aroon_period"])

uncm = all_results[best_strategy]["metrics"]
ftm = ftmo_result["metrics"]
comparison = pd.DataFrame({
    "Unconstrained": {
        "Total Return %": uncm["total_return_pct"], "Max DD %": uncm["max_drawdown_pct"],
        "Sharpe": uncm["sharpe_ratio"], "# Trades": uncm["num_trades"],
        "Win Rate %": uncm["win_rate_pct"],
    },
    "FTMO Constrained": {
        "Total Return %": ftm["total_return_pct"], "Max DD %": ftm["max_drawdown_pct"],
        "Sharpe": ftm["sharpe_ratio"], "# Trades": ftm["num_trades"],
        "Win Rate %": ftm["win_rate_pct"],
    },
})
print(f"{'='*70}")
print(f"FTMO-Constrained vs Unconstrained: {best_strategy}")
print(f"{'='*70}")
display(comparison)

# FTMO equity curve plot
fig, ax = plt.subplots(figsize=(CONFIG["fig_width"], 5))
ax.plot(all_results[best_strategy]["equity_curve"].index,
        all_results[best_strategy]["equity_curve"],
        color="#3498db", linewidth=1.2, label="Unconstrained", alpha=0.7)
ax.plot(ftmo_result["equity_curve"].index, ftmo_result["equity_curve"],
        color="#e74c3c", linewidth=1.5, label="FTMO Constrained")
ax.axhline(CONFIG["initial_capital"] * 1.10, color="green", linestyle="--",
           linewidth=0.8, label="Profit Target (+10%)")
ax.axhline(CONFIG["initial_capital"] * 0.90, color="red", linestyle="--",
           linewidth=0.8, label="Max DD Limit (-10%)")
ax.set_ylabel("Equity ($)")
ax.set_xlabel("Date")
ax.set_title(f"{best_strategy} -- FTMO Constrained Equity Curve")
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))
plt.tight_layout()
plt.show()


---
## 9. Summary & Actionable Insights


In [ ]:
# ==============================================================================
# EXECUTIVE SUMMARY
# ==============================================================================

print("=" * 80)
print("AROON INDICATOR STRATEGY BACKTESTING -- EXECUTIVE SUMMARY")
print("=" * 80)

print(f"\nUniverse: {', '.join(CONFIG['tickers'])}")
print(f"Period: {CONFIG['start_date']} to {CONFIG['end_date']}")
print(f"AROON Period: {CONFIG['aroon_period']} | Capital: ${CONFIG['initial_capital']:,.0f}")
print(f"Costs: {CONFIG['slippage_bps']} bps slippage per side")

print(f"\n{'_'*80}")
print("STRATEGY RANKING (by Sharpe on primary asset):")
print(f"{'_'*80}")

ranking = sorted(all_results.items(),
                 key=lambda x: x[1]["metrics"]["sharpe_ratio"], reverse=True)
for rank, (name, res) in enumerate(ranking, 1):
    m = res["metrics"]
    print(f"  #{rank}  {name:12s} | Sharpe: {m['sharpe_ratio']:>6.3f} | "
          f"Return: {m['total_return_pct']:>+7.1f}% | "
          f"MaxDD: {m['max_drawdown_pct']:>6.1f}% | "
          f"WinRate: {m['win_rate_pct']:>5.1f}% | PF: {m['profit_factor']:>5.2f}")

print(f"\n{'_'*80}")
print("MULTI-ASSET GENERALIZATION:")
print(f"{'_'*80}")
print(f"  Assets tested: {len(multi_asset_df)}")
print(f"  Profitable: {(multi_asset_df['total_return_pct'] > 0).sum()} / {len(multi_asset_df)}")
print(f"  Median Sharpe: {multi_asset_df['sharpe_ratio'].median():.3f}")

print(f"\n{'_'*80}")
print("MONTE CARLO (1-year forward):")
print(f"{'_'*80}")
print(f"  Prob of profit: {mc['prob_profit']*100:.0f}%")
print(f"  Expected value: ${mc['median_terminal']:,.0f}")
print(f"  95% CI: ${mc['ci_lower']:,.0f} -- ${mc['ci_upper']:,.0f}")

print(f"\n{'_'*80}")
print("WALK-FORWARD STABILITY:")
print(f"{'_'*80}")
for name, wf_df in wf_all.items():
    if len(wf_df) > 0:
        print(f"  {name:12s} | Avg OOS Sharpe: {wf_df['oos_sharpe'].mean():.3f} | "
              f"Periods: {wf_df['best_is_period'].tolist()}")


### Actionable Recommendations

**For FTMO Challenge Trading:**

1. **Strategy Selection**: Compare Sharpe ratios above. Prioritize strategies with *consistent* walk-forward OOS results over high in-sample Sharpe with poor OOS degradation.

2. **Parameter Stability**: The optimization surface plots reveal which AROON periods produce stable results. Prefer parameters in *flat* regions of the surface -- these are robust and less likely overfit.

3. **Risk Management for FTMO**:
   - The 5% daily loss and 10% total drawdown limits are binding constraints
   - Consider reducing position size below 100% to create a buffer
   - Use the Monte Carlo 5th percentile drawdown as your worst-case planning scenario

4. **Market Regime Awareness**: The regime analysis shows which conditions favor this strategy. Avoid deploying during regime transitions.

5. **Limitations & Caveats**:
   - Backtests use closing prices; real execution has additional slippage
   - AROON is a lagging indicator -- it confirms trends rather than predicting them
   - Single-indicator strategies are inherently fragile; consider combining with volume or volatility filters
   - Past performance does not guarantee future results
   - yfinance data may contain survivorship bias and adjusted price artifacts

### Next Steps

- [ ] Combine AROON with a volume filter (e.g., OBV or AVWAP) for signal confirmation
- [ ] Test on intraday timeframes for higher-frequency FTMO strategies
- [ ] Add a volatility regime filter to toggle strategy on/off
- [ ] Implement proper position sizing (Kelly Criterion or fixed-fractional)
- [ ] Paper trade for 30+ days before deploying real capital
